# PrimeVul Score Conversion (Multiple Choice)

Converts lm-eval loglikelihood output (primevul_choice_1to2) into router training data.
Same structure as convert_dataset_7_model.ipynb (MMLU/ARC section).

Paper Eq. 2: score = softmax(log_probs)[target]  if acc == 1  else  0

In [1]:
import json
import os
import glob
import random
import pandas as pd
import numpy as np
from collections import defaultdict

random.seed(42)

output_file_path = "../../datasets/primevul/balance_8models/choice"
os.makedirs(output_file_path, exist_ok=True)

# [org, model_name, output_dir_name]
# org/model_name  →  HuggingFace model ID used as key in scores dict
# output_dir_name →  directory name under output/primevul_gen/
model_list = [
    ["codellama",      "CodeLlama-7b-Instruct-hf",        "CodeLlama-7b-Instruct"],
    ["codellama",      "CodeLlama-13b-Instruct-hf",       "CodeLlama-13b-Instruct"],
    ["deepseek-ai",    "DeepSeek-Coder-V2-Lite-Instruct", "DeepSeek-Coder-V2-Lite-Instruct"],
    ["Qwen",           "Qwen2.5-Coder-14B-Instruct",      "Qwen2.5-Coder-14B-Instruct"],
    ["bigcode",        "starcoder2-15b-instruct-v0.1",    "starcoder2-15b-instruct"],
    ["Virtue-AI-HUB",  "VulnLLM-R-7B",                   "VulnLLM-R-7B"],
    ["google",         "codegemma-7b-it",                 "CodeGemma-7B-IT"],
    ["google",         "gemma-2-9b-it",                   "Gemma-2-9B-IT"],
]

# PrimeVul (loglikelihood / multiple choice)

In [2]:
# scores_by_idx[idx][model_id] = score
# funcs_by_idx[idx]            = func text
scores_by_idx = {}
funcs_by_idx  = {}

for model_index, model_info in enumerate(model_list):
    model_pre  = model_info[0]
    model      = model_info[1]
    dir_name   = model_info[2]
    model_id   = model_pre + "/" + model

    pattern = f"../../output/primevul_choice/{dir_name}/*/samples_primevul_choice_*.jsonl"
    files = glob.glob(pattern)
    if not files:
        print(f"[skip] {model}: no samples file at {pattern}")
        continue

    n_docs = 0
    avg_score = 0.0

    with open(files[0]) as f:
        for line in f:
            if not line.strip():
                continue
            entry  = json.loads(line)

            # Use doc["idx"] as stable key across model runs
            idx    = entry["doc"]["idx"]
            resps  = entry["resps"]  
            target = int(entry["target"])  # 0 or 1
            acc    = entry["acc"]          # 1.0 if correct, 0.0 if wrong

            # Paper Eq. 2: softmax over log-probs of each choice
            log_probs   = np.array([float(resps[c][0][0]) for c in range(len(resps))])
            # print(log_probs)
            probability = np.exp(log_probs)
            probability = probability / np.sum(probability)
            score = float(probability[target]) if acc == 1.0 else 0.0

            if idx not in scores_by_idx:
                scores_by_idx[idx] = {}
            scores_by_idx[idx][model_id] = score

            if idx not in funcs_by_idx:
                funcs_by_idx[idx] = entry["doc"]["func"]

            n_docs    += 1
            avg_score += 1 if acc == 1.0 else 0

    avg_score /= n_docs if n_docs else 1
    print(f"[ok] {model:<45} docs={n_docs}  avg_score={avg_score:.4f}")

# Build output_data sorted by idx for reproducibility
output_data = [
    {"question": funcs_by_idx[idx], "scores": scores_by_idx[idx]}
    for idx in sorted(scores_by_idx.keys())
]

print(f"\nTotal docs: {len(output_data)}")

[ok] CodeLlama-7b-Instruct-hf                      docs=12008  avg_score=0.4994
[ok] CodeLlama-13b-Instruct-hf                     docs=12008  avg_score=0.5012
[ok] DeepSeek-Coder-V2-Lite-Instruct               docs=12008  avg_score=0.4942
[ok] Qwen2.5-Coder-14B-Instruct                    docs=12008  avg_score=0.5406
[ok] starcoder2-15b-instruct-v0.1                  docs=12008  avg_score=0.4659
[ok] VulnLLM-R-7B                                  docs=12008  avg_score=0.6400
[ok] codegemma-7b-it                               docs=12008  avg_score=0.4985
[ok] gemma-2-9b-it                                 docs=12008  avg_score=0.5016

Total docs: 12008


In [3]:
# Train / test split  70 / 30  (same as paper)
train_split_index = random.sample(range(len(output_data)), len(output_data))
output_data = [output_data[idx] for idx in train_split_index]

train_split = output_data[:int(0.7 * len(output_data))]
test_split  = output_data[int(0.7 * len(output_data)):]

with open(os.path.join(output_file_path, "train.json"), "w") as f:
    json.dump(train_split, f)

with open(os.path.join(output_file_path, "test.json"), "w") as f:
    json.dump(test_split, f)

print(f"train: {len(train_split)}  test: {len(test_split)}")
print(f"Saved to {output_file_path}/")

train: 8405  test: 3603
Saved to ../../datasets/primevul/balance_8models/choice/


# Get ACC (per-model accuracy on train set)

In [4]:
with open(os.path.join(output_file_path, "train.json")) as f:
    train_data = json.load(f)
with open(os.path.join(output_file_path, "test.json")) as f:
    test_data = json.load(f)
output_data = train_data + test_data

# Build score dict from model list
correct_dict = {m[0]+"/"+m[1]: 0 for m in model_list}

data_size = len(output_data)
for item in output_data:
    for key, score in item["scores"].items():
        if key in correct_dict:
            correct_dict[key] += score / data_size * 100

df = pd.DataFrame.from_dict(correct_dict, orient="index", columns=["accuracy (%)"])

# Also show how many queries have partial scores (0 < s < 1) per model
partial_dict = {m[0]+"/"+m[1]: 0 for m in model_list}
for item in output_data:
    for key, score in item["scores"].items():
        if key in partial_dict and 1e-9 < score < 1 - 1e-9:
            partial_dict[key] += 1

df["partial_scores"] = pd.Series(partial_dict)
df["partial_%"] = (df["partial_scores"] / data_size * 100).round(1)
print(f"Total samples: {data_size} (train={len(train_data)}, test={len(test_data)})")
df

Total samples: 12008 (train=8405, test=3603)


,accuracy (%),partial_scores,partial_%
codellama/CodeLlama-7b-Instruct-hf,40.648049,5997,49.9
codellama/CodeLlama-13b-Instruct-hf,40.000269,6018,50.1
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct,48.930527,5934,49.4
Qwen/Qwen2.5-Coder-14B-Instruct,49.639752,6491,54.1
bigcode/starcoder2-15b-instruct-v0.1,40.492818,5595,46.6
Virtue-AI-HUB/VulnLLM-R-7B,56.587733,7685,64.0
google/codegemma-7b-it,48.768785,5986,49.9
google/gemma-2-9b-it,47.738343,6023,50.2


In [5]:
with open(os.path.join(output_file_path, "train.json")) as f:
    train_data = json.load(f)
with open(os.path.join(output_file_path, "test.json")) as f:
    test_data = json.load(f)
output_data = train_data + test_data

M = 5   # number of runs per query

model_ids = list(output_data[0]["scores"].keys())
total_queries = len(output_data)

print(f"Total samples: {total_queries} (train={len(train_data)}, test={len(test_data)})")
print(f"{'Model':<55} {'correct':>10} {'total':>10} {'accuracy':>10}")
print("─" * 90)

for model_id in model_ids:
    # score = correct / M  →  correct = score * M
    total_correct = sum(item["scores"][model_id] * M for item in output_data)
    total_possible = total_queries * M
    accuracy = total_correct / total_possible * 100
    print(f"{model_id:<55} {total_correct:>10.0f} {total_possible:>10,} {accuracy:>9.2f}%")

Total samples: 12008 (train=8405, test=3603)
Model                                                      correct      total   accuracy
──────────────────────────────────────────────────────────────────────────────────────────
codellama/CodeLlama-7b-Instruct-hf                           24405     60,040     40.65%
codellama/CodeLlama-13b-Instruct-hf                          24016     60,040     40.00%
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct                  29378     60,040     48.93%
Qwen/Qwen2.5-Coder-14B-Instruct                              29804     60,040     49.64%
bigcode/starcoder2-15b-instruct-v0.1                         24312     60,040     40.49%
Virtue-AI-HUB/VulnLLM-R-7B                                   33975     60,040     56.59%
google/codegemma-7b-it                                       29281     60,040     48.77%
google/gemma-2-9b-it                                         28662     60,040     47.74%
